# 完整的 GPT 模型

各组件已就绪：分词器、嵌入、RoPE 位置编码、
注意力、以及打包成可重复单元的 Transformer 块。

现在组装成单一模型：输入 token ID，预测下一个 token。
这与 GPT-2、GPT-3、LLaMA、Mistral 的架构相同。

模型四部分：token 嵌入（ID→向量）；
Transformer 块堆栈（注意力+前馈）；最终归一化；
输出头（隐状态→词表分数）。

技巧是权重绑定（weight tying）：
嵌入表与输出头共享权重，省约三分之一参数，
且每个 token 嵌入从两个方向获得梯度，训练信号更强。

读完本 notebook，你将拥有一个可训练的完整语言模型。

## 导入

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

## 来自前面章节的构建块

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)


def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super().__init__()
        hidden_dim = expansion_factor * d_model
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = self.rotary(q, seq_len)
        k = self.rotary(k, seq_len)
        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        attn_output = attn_weights @ v
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)
        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

## GPT 配置

一个 dataclass 包含全部设置，改一个数整模适配。

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 50257
    d_model: int = 768
    num_heads: int = 12
    num_layers: int = 12
    max_seq_len: int = 1024
    dropout: float = 0.1
    embd_dropout: float = 0.1
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    warmup_steps: int = 2000
    max_steps: int = 100000
    batch_size: int = 8
    grad_accum_steps: int = 4
    betas: tuple = (0.9, 0.95)
    eps: float = 1e-8

## GPT 模型

Token 嵌入 + dropout；Transformer 块堆栈；
最终 RMSNorm；投影到词表分数。
嵌入与输出头共享权重；
所有权重从 std=0.02 的正态分布初始化。

In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.embd_dropout = nn.Dropout(config.embd_dropout)

        self.layers = nn.ModuleList([
            TransformerBlock(config.d_model, config.num_heads, config.dropout)
            for _ in range(config.num_layers)
        ])

        self.final_norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if hasattr(module, 'bias') and module.bias is not None:
                torch.nn.init.zeros_(module.bias)

    def forward(self, input_ids, targets=None):
        batch_size, seq_len = input_ids.shape

        x = self.token_embedding(input_ids)
        x = self.embd_dropout(x)

        mask = create_causal_mask(seq_len, input_ids.device)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_norm(x)

        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            logits_flat = logits[:, :-1, :].contiguous().view(-1, self.config.vocab_size)
            targets_flat = targets[:, 1:].contiguous().view(-1)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

## 创建模型并查看

In [ ]:
config = GPTConfig()
model = GPT(config)
print(f"总参数量: {model.get_num_params():,}")
print()

print(f"配置:")
print(f"  d_model:   {config.d_model}")
print(f"  num_heads: {config.num_heads}")
print(f"  num_layers:{config.num_layers}")
print(f"  vocab_size:{config.vocab_size}")
print(f"  max_seq_len: {config.max_seq_len}")

## 前向传播

输入随机 token ID，检查 logits 形状是否正确，并计算 loss。

In [ ]:
batch_size = 4
seq_len = 128

input_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))
target_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))

logits, loss = model(input_ids, target_ids)

print(f"输入形状:  {input_ids.shape}")
print(f"Logits 形状: {logits.shape}")
print(f"损失:         {loss.item():.4f}")
print()

print(f"期望 logits 形状: [{batch_size}, {seq_len}, {config.vocab_size}]")
print(f"实际 logits 形状:   [{logits.shape[0]}, {logits.shape[1]}, {logits.shape[2]}]")
print()

print(f"每个位置（共 {seq_len} 个）模型输出")
print(f"{config.vocab_size} 个分数，对应每个可能的下一个 token。")

## 验证权重绑定

嵌入与输出头共享同一张量，改一处即改另一处，因指向同一内存。

In [ ]:
emb_weight = model.token_embedding.weight
head_weight = model.lm_head.weight

print(f"嵌入权重 id: {id(emb_weight)}")
print(f"LM head 权重 id:   {id(head_weight)}")
print(f"同一内存:         {emb_weight.data_ptr() == head_weight.data_ptr()}")
print()

saved_params = config.vocab_size * config.d_model
print(f"权重绑定节省的参数量: {saved_params:,}")